# GSEA for IntAct Complex Portal using Network Centrality
**Objective:** This notebook performs a Gene Set Enrichment Analysis (GSEA) to identify protein complexes from the Complex Portal that are significantly enriched for proteins with high (or low) network centrality.

**Workflow:**

  **Step 1: Setup:** Install and import necessary Python libraries.

  **Step 2: Prepare Ranked Gene List (.rnk):** Combine the centrality scores from the C++ program with the ID mapping file to create a GSEA-compatible ranked list based on UniProt IDs.

  **Step 3: Prepare Gene Set Database (.gmt):** Parse original `complex.tsv` file to create a gene set library of all human complexes.

  **Step 4: Run GSEA:** Execute the analysis using the `gseapy` library.

  **Step 5: Diagnose:** Check for common issues like mismatched gene identifiers to ensure the analysis ran correctly.

# Setup Environment and Import Libraries

In [1]:
!pip install gseapy -q

import pandas as pd
import gseapy as gp
import os
from google.colab import files

print("Libraries imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 597.6/597.6 kB 12.8 MB/s eta 0:00:00
Libraries imported successfully.


# Prepare Ranked Gene List (.rnk file)

Combine centrality scores and ID mapping file.

**Action:** Run this cell and upload the following two files:

1.  The betweenness score file from your C++ program (e.g., `complexportal_dataset.ungraph.betweenness.txt`).

2.  The ID mapping file you created earlier (e.g., `complexportal_dataset_id_mapping.tsv`).

In [3]:
# --- Configuration ---
centrality_input_file = "/content/ebi-intact.ungraph.betweenness.txt"  # Change to your actual file name/path
mapping_input_file = "/content/id_mapping.tsv"         # Change to your actual file name/path
rnk_output_file = "complexportal_betweenness.rnk"

# Check if files exist
if not os.path.isfile(centrality_input_file) or not os.path.isfile(mapping_input_file):
    print("\nERROR: One or both required files were not found at the specified path.")
    print(f"Checked for:\n  Centrality file: {centrality_input_file}\n  Mapping file: {mapping_input_file}")
else:
    print(f"\nFound centrality file: '{centrality_input_file}'")
    print(f"Found mapping file: '{mapping_input_file}'")

    # --- Processing ---
    try:
        # Read the centrality data (integer_id, score), skipping header lines if any
        df_scores = pd.read_csv(centrality_input_file, sep=' ', comment='#', header=None, skiprows=2)
        df_scores.columns = ['Integer_ID', 'Score']

        # Read the ID mapping data (integer_id, participant_id)
        df_map = pd.read_csv(mapping_input_file, sep='\t')
        df_map.columns = ['Integer_ID', 'Participant_ID']

        # Merge the two dataframes to link scores to UniProt IDs
        df_merged = pd.merge(df_scores, df_map, on='Integer_ID')

        # Create the final dataframe for the .rnk file
        df_rnk = df_merged[['Participant_ID', 'Score']]

        # Sort by score in descending order (highest centrality at the top)
        df_rnk = df_rnk.sort_values(by='Score', ascending=False)

        # Save to the GSEA-ready .rnk file
        df_rnk.to_csv(rnk_output_file, sep='\t', header=False, index=False)

        print(f"\nSuccessfully created GSEA-ready ranked list: {rnk_output_file}")
        print("\nFirst 5 lines of the .rnk file:")
        !head -n 5 {rnk_output_file}

    except Exception as e:
        print(f"\nAn error occurred during processing: {e}")
        print("Please check the format of your input files.")



Found centrality file: '/content/ebi-intact.ungraph.betweenness.txt'
Found mapping file: '/content/id_mapping.tsv'

Successfully created GSEA-ready ranked list: complexportal_betweenness.rnk

First 5 lines of the .rnk file:
CHEBI:29105	347344.834917945
P55769	246347.866666666
Q9NV06	218160.043478261
P62877	211749.460144928
P61964	178258.70994769


# Prepare Gene Set Database (.gmt file)

Parse original `complex.tsv` file to create a gene set library for GSEA.

**Action:** Run this cell and upload your original `complex.tsv` file.

In [4]:
# --- Configuration ---
complex_portal_input_file = "/content/complex.tsv"  # Adjust path as needed
gmt_output_file = "complexportal_human.gmt"

# Check if file exists
if not os.path.isfile(complex_portal_input_file):
    print(f"\nERROR: File '{complex_portal_input_file}' not found. Please place the file at the specified path.")
else:
    print(f"\nFound file: '{complex_portal_input_file}'")

    # --- Processing Function ---
    def create_complexportal_gmt(input_tsv_path, output_gmt_path):
        try:
            # Read the tsv, skipping the first line which is a comment
            df = pd.read_csv(input_tsv_path, sep='\t', header=0, skiprows=1)

            # Assign column names for clarity
            df.columns = [
                'Complex_AC', 'Recommended_Name', 'Aliases', 'Taxonomy_ID',
                'Participants', 'Evidence_Code', 'Experimental_Evidence',
                'GO_Annotations', 'Cross_References', 'Description', 'Properties',
                'Assembly', 'Ligand', 'Disease', 'Agonist', 'Antagonist',
                'Comment', 'Source', 'Expanded_Participants'
            ]

            # Filter for human complexes (Taxonomy ID for Homo sapiens is 9606)
            df_human = df[df['Taxonomy_ID'] == 9606].copy()
            print(f"Found {len(df_human)} human complexes in the input file.")

            complex_count = 0
            with open(output_gmt_path, 'w') as gmt_file:
                for index, row in df_human.iterrows():
                    complex_name = row['Recommended_Name']
                    description = row['Complex_AC']
                    participants_str = row['Participants']

                    if isinstance(participants_str, str):
                        # Parse: 'P123(1)|Q456(2)' → ['P123', 'Q456']
                        participants_list = [
                            p.split('(')[0] for p in participants_str.split('|')
                        ]
                        unique_participants = sorted(list(set(participants_list)))

                        if unique_participants:
                            gmt_file.write(f"{complex_name}\t{description}\t" + "\t".join(unique_participants) + "\n")
                            complex_count += 1

            print(f"\nSuccessfully created GMT file with {complex_count} human complexes at: {output_gmt_path}")
            print("\nFirst 3 lines of the .gmt file:")
            !head -n 3 {output_gmt_path}

        except Exception as e:
            print(f"\nAn error occurred: {e}")

    # --- Run the function ---
    create_complexportal_gmt(complex_portal_input_file, gmt_output_file)



Found file: '/content/complex.tsv'
Found 2345 human complexes in the input file.

Successfully created GMT file with 2345 human complexes at: complexportal_human.gmt

First 3 lines of the .gmt file:
bZIP transcription factor complex, ATF4-CREB1	CPX-8	P16220	P18848
bZIP transcription factor complex, ATF1-ATF4	CPX-9	P18846	P18848
SMAD2 homotrimer	CPX-11	Q15796


# Step 4: Execute GSEA Pre-Ranked Analysis

Run the core GSEA analysis using the files created in the previous steps.

In [5]:
# --- Configuration ---
# These should match the output files from the previous steps.
rnk_file = "complexportal_betweenness.rnk"
gmt_file = "complexportal_human.gmt"
output_directory = "gsea_results_complexportal"

# --- Run GSEA Pre-Ranked ---
print("Starting GSEA Preranked analysis...")
try:
    # Check if the input files exist before running
    if not os.path.exists(rnk_file) or not os.path.exists(gmt_file):
      print(f"❌ ERROR: One of the input files ('{rnk_file}' or '{gmt_file}') is missing.")
      print("Please ensure the previous steps ran successfully.")
    else:
      pre_res = gp.prerank(
          rnk=rnk_file,
          gene_sets=gmt_file,
          outdir=output_directory,
          min_size=3,      # Min size of a complex to be tested
          max_size=500,    # Max size of a complex to be tested
          permutation_num=1000,  # Number of permutations for significance testing
          format='png',    # Output plot format
          seed=42,         # For reproducible results
          verbose=True     # Show progress
      )

      print(f"\nAnalysis complete! Results are saved in the '{output_directory}' folder.")
      print("You can view and download the result files from the file browser on the left.")
      print("\nDisplaying the top enriched terms:")
      display(pre_res.res2d.head(10))


except Exception as e:
    print(f"\nAn error occurred during GSEA analysis: {e}")
    print("Please check that your .rnk and .gmt files were created successfully and are not empty.")

2025-07-31 07:42:39,454 [WARNING] Duplicated values found in preranked stats: 90.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2025-07-31 07:42:39,455 [INFO] Parsing data files for GSEA.............................
2025-07-31 07:42:39,471 [INFO] 1000 gene_sets have been filtered out when max_size=500 and min_size=3
2025-07-31 07:42:39,472 [INFO] 1343 gene_sets used for further statistical testing.....
2025-07-31 07:42:39,473 [INFO] Start to run GSEA...Might take a while..................


Starting GSEA Preranked analysis...


2025-07-31 07:43:20,164 [INFO] Congratulations. GSEApy runs successfully................




Analysis complete! Results are saved in the 'gsea_results_complexportal' folder.
You can view and download the result files from the file browser on the left.

Displaying the top enriched terms:


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,MKS transition zone complex,-0.929127,-2.269518,0.0,0.0,0.0,14/14,100.00%,Q96Q45;Q9BPU9;Q9H6L2;Q9NXB0;Q9P0N5;Q86X19;Q96G...
1,prerank,MCM complex,-0.935385,-2.093758,0.0,0.0,0.0,6/6,100.00%,P25205;P33991;P33992;Q14566;P33993;P49736
2,prerank,Signal recognition particle,-0.92233,-2.080354,0.0,0.0,0.0,7/7,100.00%,URS00000478B7_9606;P61011;P49458;Q9UHB9;O76094...
3,prerank,Tubulin polyglutamylase complex,-0.933463,-2.059316,0.0,0.0,0.0,5/5,100.00%,O95922;Q6ZTW0;Q8IUZ0;Q9BSH3;Q68CL5
4,prerank,Outer dynein arm-docking complex,-0.93374,-2.048438,0.0,0.0,0.0,5/5,100.00%,Q96NG3;Q9HAE3;A5D8V7;Q5T2S8;Q96M63
5,prerank,Synaptonemal complex,-0.900971,-2.042518,0.0,0.0,0.0,7/7,100.00%,Q9BXU0;A1L190;Q15431;Q6PIF2;Q8IZU3;Q8N0S2;Q9BX26
6,prerank,FERRY RAB5 effector complex,-0.953978,-2.00237,0.0,0.0,0.0,5/5,100.00%,O95825;Q6ZMI0;Q8NB37;Q8TEA7;Q9NQ89
7,prerank,USH2 complex,-0.965909,-1.98412,0.0,0.000143,0.001,4/4,100.00%,O75445;Q8WXG9;Q9H5P4;Q9P202
8,prerank,ESCRT-III complex,-0.801722,-1.962388,0.0,0.000254,0.003,11/11,100.00%,Q9Y3E7;Q8WUX9;Q96CF2;Q96FZ7;Q9H444;Q9HD42;Q9NZ...
9,prerank,UTP-A complex,-0.893511,-1.951399,0.0,0.000343,0.005,6/6,100.00%,Q8IWA0;Q8TED0;Q969X6;Q9H583;Q9H8H0;Q15061


# Diagnose Gene ID Mismatch (Important Sanity Check)

This script checks the overlap between the gene/protein identifiers in your `.rnk` file and your `.gmt` file. A low overlap is the most common reason for getting no significant results.


Since both files were generated from the same source data, the overlap should be very high.

In [6]:
rnk_file = "complexportal_betweenness.rnk"
gmt_file = "complexportal_human.gmt"

try:
    # --- Read genes from the .rnk file ---
    ranked_genes_df = pd.read_csv(rnk_file, sep='\t', header=None, usecols=[0])
    ranked_genes = set(ranked_genes_df[0].str.strip())
    print(f"Found {len(ranked_genes)} unique identifiers in your ranked list (.rnk file).")
    print(f"Examples from .rnk: {list(ranked_genes)[:5]}\n")

    # --- Read genes from the .gmt file ---
    gmt_genes = set()
    with open(gmt_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            for gene in parts[2:]:
                gmt_genes.add(gene.strip())
    print(f"Found {len(gmt_genes)} unique identifiers across all complexes (.gmt file).")
    print(f"Examples from .gmt: {list(gmt_genes)[:5]}\n")

    # --- Calculate and report the overlap ---
    overlapping_genes = ranked_genes.intersection(gmt_genes)

    print("--- DIAGNOSIS ---")
    print(f"Number of overlapping identifiers found in both files: {len(overlapping_genes)}")

    if len(ranked_genes) > 0:
      # This is the crucial check: how many of the genes you ranked could possibly be found?
      overlap_percentage = (len(overlapping_genes) / len(ranked_genes)) * 100
      print(f"Overlap percentage: {overlap_percentage:.2f}% of your ranked identifiers are present in the GMT file.")

    if overlap_percentage < 90:
        print("\nCONCLUSION: WARNING - Overlap is lower than expected!")
        print("There might be a discrepancy in how the files were generated.")
    else:
        print("\nCONCLUSION: SUCCESS! The identifier overlap is excellent.")
        print("This confirms that the input files for GSEA are consistent.")

except FileNotFoundError as e:
    print(f"ERROR: A file was not found. Please ensure '{rnk_file}' and '{gmt_file}' exist. Error: {e}")

Found 3612 unique identifiers in your ranked list (.rnk file).
Examples from .rnk: ['O60494', 'Q7LG56', 'P62249', 'Q9Y251-PRO_0000042262', 'P10244']

Found 3693 unique identifiers across all complexes (.gmt file).
Examples from .gmt: ['O60494', 'O95460', 'Q7LG56', 'P62249', 'Q9Y251-PRO_0000042262']

--- DIAGNOSIS ---
Number of overlapping identifiers found in both files: 3612
Overlap percentage: 100.00% of your ranked identifiers are present in the GMT file.

CONCLUSION: SUCCESS! The identifier overlap is excellent.
This confirms that the input files for GSEA are consistent.
